# Notebook 01 — Dataset Overview

This notebook walks through the Polymarket dataset and the full data pipeline, from raw SQLite tables to the feature matrices used by the three trading strategies.

**Contents:**
1. Raw database schema and audit
2. Usability funnel attrition (58k → 14.5k markets)
3. Market distributions (volume, spread, trade frequency)
4. Volume concentration (Gini, Lorenz curve)
5. LLM semantic categories
6. Train / Test split


In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

ROOT = Path("..")
sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed"
FEATURES  = ROOT / "data" / "features"

plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Setup OK")

## 1. Audit Report

In [ ]:
with open(PROCESSED / "audit_report.json") as f:
    audit = json.load(f)

ov = audit["overview"]
print(f"Total markets in database:  {ov['n_markets_unique']:,}")
print(f"Total 1-min bars:           {ov.get('n_bars_total', 'N/A'):,}")
print(f"Date range:                 {ov.get('date_min', '')} → {ov.get('date_max', '')}")

## 2. Funnel Attrition

In [ ]:
with open(PROCESSED / "funnel_attrition.json") as f:
    funnel = json.load(f)["attrition"]

df_funnel = pd.DataFrame(funnel)
df_funnel[["step", "label", "n_after", "pct_removed"]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(df_funnel["label"], df_funnel["n_after"], color="#3B82F6", height=0.6)
for i, (n, p) in enumerate(zip(df_funnel["n_after"], df_funnel["pct_removed"])):
    ax.text(n + 200, i, f"{n:,}  (−{p:.1f}%)", va="center", fontsize=9)
ax.set_xlabel("Markets remaining")
ax.set_title("Usability Funnel — Polymarket Markets", fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Market Distributions

In [ ]:
df_pm = pd.read_parquet(PROCESSED / "per_market_usable.parquet")
print(f"Usable markets: {len(df_pm):,}")
df_pm[["total_volume_usdc", "trade_count", "duration_days"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (col, label, log) in zip(axes, [
    ("total_volume_usdc", "Total Volume (USDC)", True),
    ("trade_count",       "Trade Count",         True),
    ("duration_days",     "Market Duration (days)", False),
]):
    data = df_pm[col].dropna()
    if log:
        data = np.log10(data.clip(lower=1))
        label = f"log₁₀({label})"
    ax.hist(data, bins=50, color="#6366F1", edgecolor="white", lw=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(label, fontsize=10)

plt.suptitle("Per-Market Distribution — Usable Markets", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 4. Volume Concentration

In [ ]:
with open(PROCESSED / "concentration_report.json") as f:
    conc = json.load(f)

mc = conc["market_concentration"]
print(f"Gini coefficient (volume): {mc['gini_volume']:.4f}")
print(f"Top 0.1% volume share:     {mc['top01_pct_volume']:.1f}%")
print(f"Top 1% volume share:       {mc['top1_pct_volume']:.1f}%")
print(f"Top 10% volume share:      {mc['top10_pct_volume']:.1f}%")

In [ ]:
# Lorenz curve
if "lorenz" in conc:
    lorenz = conc["lorenz"]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(lorenz["pop_share"], lorenz["vol_share"], color="#3B82F6", lw=2, label="Volume Lorenz")
    ax.plot([0, 1], [0, 1], "--", color="grey", label="Perfect equality")
    ax.fill_between(lorenz["pop_share"], lorenz["vol_share"], lorenz["pop_share"],
                    alpha=0.15, color="#3B82F6")
    ax.set_xlabel("Cumulative share of markets")
    ax.set_ylabel("Cumulative share of volume")
    ax.set_title(f"Lorenz Curve — Gini = {mc['gini_volume']:.3f}", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. LLM Category Distribution

In [ ]:
df_feat = pd.read_parquet(FEATURES / "features_1m_long.parquet",
                          columns=["llm_category"])
cat_counts = df_feat["llm_category"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
cat_counts[::-1].plot.barh(ax=ax, color="#10B981", width=0.7)
ax.set_xlabel("Number of 1-min bars")
ax.set_title("Top 20 LLM Categories by Activity", fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Train / Test Split

In [ ]:
TRAIN_END = "2025-11-01"

for split in ["train", "test"]:
    df = pd.read_parquet(FEATURES / f"features_1m_{split}_wide.parquet", columns=["minute_ts"])
    df["minute_ts"] = pd.to_datetime(df["minute_ts"])
    print(f"{split.capitalize()}: {df['minute_ts'].min().date()} → {df['minute_ts'].max().date()}  "
          f"({len(df):,} rows)")